In [38]:
!pip install xlrd
import pandas as pd 
!pip install unidecode thefuzz


In [22]:
df = pd.read_csv("REGISTROS_com_categoria_2025_utf8.csv")

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19184 entries, 0 to 19183
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ANO_EMISSAO      19184 non-null  int64 
 1   MES_MISSAO       19184 non-null  int64 
 2   UF               19184 non-null  object
 3   MUNICIPIO        19184 non-null  object
 4   STATUS_REGISTRO  19184 non-null  object
 5   ESPECIE_ARMA     19184 non-null  object
 6   MARCA_ARMA       19184 non-null  object
 7   CALIBRE_ARMA     19184 non-null  object
 8   SEXO             19184 non-null  object
 9   CATEGORIA        19184 non-null  object
 10  TOTAL            19184 non-null  int64 
dtypes: int64(3), object(8)
memory usage: 1.6+ MB


In [30]:
path = "estimativa_dou_2025.xls"
xls = pd.ExcelFile(path, engine="xlrd")   
print(xls.sheet_names)

['BRASIL E UFs', 'Municípios']


In [33]:
df_mun = pd.read_excel(path, sheet_name="Municípios", engine="xlrd")

In [35]:
df_mun = pd.read_excel(path, sheet_name="BRASIL E UFs", engine="xlrd")

In [41]:
df_mun = df_mun  

# 1) inspeciona colunas reais
print("Colunas do DataFrame (com índices):")
for i, c in enumerate(df_mun.columns):
    print(i, repr(c))

# 2) função utilitária para achar colunas por lista de candidatos
def find_col(df, candidates):
    cols = df.columns.tolist()
    for cand in candidates:
        # busca exata ignorando espaços/maiusculas/acentos
        for c in cols:
            if unidecode(str(c)).strip().upper() == unidecode(cand).strip().upper():
                return c
    # busca por substring relaxada
    for cand in candidates:
        for c in cols:
            if unidecode(c).strip().upper().find(unidecode(cand).strip().upper()) >= 0:
                return c
    return None

# 3) candidatos comuns 
candidatos_cod = ["CODIGO MUNICIPIO", "CODIGO MUNICÍPIO", "COD_MUNICIPIO", "COD_IBGE", "CÓDIGO MUNICÍPIO", "CODIGO"]
candidatos_mun = ["MUNICIPIO", "MUNICÍPIO", "NOME MUNICIPIO", "NOME", "MUNicípio", "Município"]
candidatos_uf = ["UF", "ESTADO", "SIGLA"]
candidatos_pop = ["POPULACAO", "POPULAÇÃO", "ESTIMATIVA", "POPULAÇÃO ESTIMADA", "POPULACAO ESTIMADA"]

col_cod = find_col(df_mun, candidatos_cod)
col_mun = find_col(df_mun, candidatos_mun)
col_uf  = find_col(df_mun, candidatos_uf)
col_pop = find_col(df_mun, candidatos_pop)

print("\nColunas detectadas (pode haver None):")
print("cod_ibge ->", col_cod)
print("municipio ->", col_mun)
print("uf ->", col_uf)
print("populacao ->", col_pop)

# 4) se algum for None, imprime as primeiras linhas para ajudar a identificar
if not all([col_mun, col_uf, col_pop]):
    print("\nMostrando primeiras 8 linhas (header + 8) para inspeção manual:")
    display(df_mun.head(8))

# 5) renomeia para nomes padrão e aplica limpeza (só se detectou os nomes)
def normalize_text(s):
    if pd.isna(s): return s
    return unidecode(str(s)).strip().upper()

if col_mun and col_uf and col_pop:
    df_clean = df_mun.rename(columns={col_mun: "municipio", col_uf: "uf", col_pop: "populacao"})
    if col_cod:
        df_clean = df_clean.rename(columns={col_cod: "cod_ibge"})
    # aplicar normalização
    df_clean["municipio"] = df_clean["municipio"].astype(str).apply(normalize_text)
    df_clean["uf"] = df_clean["uf"].astype(str).str.strip().str.upper()
    if "cod_ibge" in df_clean.columns:
        df_clean["cod_ibge"] = pd.to_numeric(df_clean["cod_ibge"].astype(str).str.extract(r"(\d+)")[0], errors="coerce").astype("Int64")
    df_clean["populacao"] = pd.to_numeric(df_clean["populacao"].astype(str).str.replace(r"[^0-9]", "", regex=True), errors="coerce").fillna(0).astype(int)
    df_clean["municipio_clean"] = df_clean["municipio"]
    print("\nDataFrame limpo — exemplos:")
    display(df_clean[["cod_ibge","municipio_clean","uf","populacao"]].head())
else:
    print("\nNão foi possível detectar todas as colunas necessárias automaticamente")

Colunas do DataFrame (com índices):
0 'ESTIMATIVAS DA POPULAÇÃO RESIDENTE NO BRASIL E UNIDADES DA FEDERAÇÃO COM DATA DE REFERÊNCIA EM 1º DE JULHO DE 2025'
1 'Unnamed: 1'
2 'Unnamed: 2'

Colunas detectadas (pode haver None):
cod_ibge -> None
municipio -> None
uf -> None
populacao -> ESTIMATIVAS DA POPULAÇÃO RESIDENTE NO BRASIL E UNIDADES DA FEDERAÇÃO COM DATA DE REFERÊNCIA EM 1º DE JULHO DE 2025

Mostrando primeiras 8 linhas (header + 8) para inspeção manual:


,ESTIMATIVAS DA POPULAÇÃO RESIDENTE NO BRASIL E UNIDADES DA FEDERAÇÃO COM DATA DE REFERÊNCIA EM 1º DE JULHO DE 2025,Unnamed: 1,Unnamed: 2
0,BRASIL E UNIDADES DA FEDERAÇÃO,POPULAÇÃO ESTIMADA,NaN
1,Brasil,213421037,NaN
2,Norte,18801282,NaN
3,Rondônia,1751950,NaN
4,Acre,884372,NaN
5,Amazonas,4321616,NaN
6,Roraima,738772,NaN
7,Pará,8711196,NaN



Não foi possível detectar todas as colunas necessárias automaticamente. Veja a saída acima e me diga qual nome corresponde a 'município', 'uf', 'população' ou 'cod_ibge' — ou copie aqui o print das colunas.


In [42]:
df_mun = pd.read_excel(path, sheet_name="Municípios", engine="xlrd")
df_mun.head(10)


,ESTIMATIVAS DA POPULAÇÃO RESIDENTE NOS MUNICÍPIOS BRASILEIROS COM DATA DE REFERÊNCIA EM 1º DE JULHO DE 2025,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
1,RO,11,00015,Alta Floresta D'Oeste,22787
2,RO,11,00023,Ariquemes,109170
3,RO,11,00031,Cabixi,5664
4,RO,11,00049,Cacoal,98280
5,RO,11,00056,Cerejeiras,16966
6,RO,11,00064,Colorado do Oeste,16508
7,RO,11,00072,Corumbiara,7968
8,RO,11,00080,Costa Marques,13510
9,RO,11,00098,Espigão D'Oeste,32842


In [43]:
path = "estimativa_dou_2025.xls"   
sheet = "Municípios"               

# 1) Ler usando a segunda linha (header=1)
df = pd.read_excel(path, sheet_name=sheet, engine="xlrd", header=1)

# 2) Mostrar colunas detectadas para conferência
print("Colunas detectadas:", df.columns.tolist())
display(df.head(6))

# 3) Renomear colunas esperadas
rename_map = {
    "UF": "uf",
    "COD. UF": "cod_uf",
    "COD. MUNIC": "cod_mun",
    "NOME DO MUNICÍPIO": "municipio",
    "POPULAÇÃO ESTIMADA": "populacao"
}
# aplica renomeação aproximada: procura colunas por match exato ignorando espaços/maiusculas/acentos
def find_and_rename(cols, mapping):
    newmap = {}
    for c in cols:
        for k,v in mapping.items():
            if unidecode(str(c)).strip().upper() == unidecode(k).strip().upper():
                newmap[c] = v
    return newmap

df = df.rename(columns=find_and_rename(df.columns, rename_map))

# 4) Limpeza/normalização
def normalize_text(s):
    if pd.isna(s): return s
    return unidecode(str(s)).strip().upper()

# garantir colunas obrigatórias existem
expected = ["uf","cod_uf","cod_mun","municipio","populacao"]
missing = [c for c in expected if c not in df.columns]
if missing:
    raise SystemExit(f"Colunas faltando após renomear: {missing}. Verifique nomes reais: {df.columns.tolist()}")

# normalizar
df["uf"] = df["uf"].astype(str).str.strip().str.upper()
df["cod_uf"] = df["cod_uf"].astype(str).str.replace(r"[^0-9]", "", regex=True).str.zfill(2)
df["cod_mun"] = df["cod_mun"].astype(str).str.replace(r"[^0-9]", "", regex=True).str.zfill(5)
# cod_ibge: UF (2 dig) + MUNIC (5 dig) => 7 dígitos
df["cod_ibge"] = (df["cod_uf"].fillna("00") + df["cod_mun"].fillna("00000")).replace(r"^0+$", pd.NA, regex=True)
df["cod_ibge"] = pd.to_numeric(df["cod_ibge"], errors="coerce").astype("Int64")

# municipio clean
df["municipio"] = df["municipio"].astype(str)
df["municipio_clean"] = df["municipio"].apply(lambda s: normalize_text(s))

# populacao -> int
df["populacao"] = pd.to_numeric(df["populacao"].astype(str).str.replace(r"[^0-9]", "", regex=True), errors="coerce").fillna(0).astype(int)

# 5) Remover linhas que são título ou vazias (ex.: se municipio for 'UF' ou 'BRASIL' etc)
# Filtro básico: manter linhas onde municipio_clean não é vazio e não é 'UF' nem 'BRASIL'
bad_mask = df["municipio_clean"].isin(["", "UF", "BRASIL", "UNIDADES DA FEDERACAO"])
df_clean = df.loc[~bad_mask].copy()

# 6) Checagens rápidas
print("Tamanho original:", len(df), "-> após limpeza:", len(df_clean))
print("Exemplos (após limpeza):")
display(df_clean[["cod_ibge","uf","municipio","municipio_clean","populacao"]].head(10))


no_cod = df_clean["cod_ibge"].isna().sum()
print("Registros sem cod_ibge:", no_cod)

# 7) salvar CSV limpo pronto para merge (utf-8-sig para Excel compat)
out_dir = Path("data_clean")
out_dir.mkdir(exist_ok=True)
csv_path = out_dir / "estimativa_municipios_2025_clean.csv"
df_clean.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("CSV salvo em:", csv_path)


Colunas detectadas: ['UF', 'COD. UF', 'COD. MUNIC', 'NOME DO MUNICÍPIO', 'POPULAÇÃO ESTIMADA']


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
0,RO,11.0,15.0,Alta Floresta D'Oeste,22787.0
1,RO,11.0,23.0,Ariquemes,109170.0
2,RO,11.0,31.0,Cabixi,5664.0
3,RO,11.0,49.0,Cacoal,98280.0
4,RO,11.0,56.0,Cerejeiras,16966.0
5,RO,11.0,64.0,Colorado do Oeste,16508.0


Tamanho original: 5573 -> após limpeza: 5573
Exemplos (após limpeza):


,cod_ibge,uf,municipio,municipio_clean,populacao
0,11000150,RO,Alta Floresta D'Oeste,ALTA FLORESTA D'OESTE,227870
1,11000230,RO,Ariquemes,ARIQUEMES,1091700
2,11000310,RO,Cabixi,CABIXI,56640
3,11000490,RO,Cacoal,CACOAL,982800
4,11000560,RO,Cerejeiras,CEREJEIRAS,169660
5,11000640,RO,Colorado do Oeste,COLORADO DO OESTE,165080
6,11000720,RO,Corumbiara,CORUMBIARA,79680
7,11000800,RO,Costa Marques,COSTA MARQUES,135100
8,11000980,RO,Espigão D'Oeste,ESPIGAO D'OESTE,328420
9,11001060,RO,Guajará-Mirim,GUAJARA-MIRIM,435940


Registros sem cod_ibge: 2
CSV salvo em: data_clean\estimativa_municipios_2025_clean.csv


In [45]:
path = "estimativa_dou_2025.xls"
sheet = "Municípios"

# 1) ler com header=1
df = pd.read_excel(path, sheet_name=sheet, engine="xlrd", header=1)

# 2) inspeciona colunas reais
print("Colunas originais:", df.columns.tolist())
display(df.head(6))

# 3) função para normalizar strings de comparação
def norm_str(s):
    if pd.isna(s): 
        return ""
    return unidecode(str(s)).upper().strip()

# 4) mapeamento aproximado para renomear colunas
candidates_map = {
    "UF": "uf",
    "COD. UF": "cod_uf",
    "COD. MUNIC": "cod_mun",
    "NOME DO MUNICÍPIO": "municipio",
    "POPULAÇÃO ESTIMADA": "populacao_raw"
}

# detectar quais colunas do df correspondem aos candidatos
rename_map = {}
for col in df.columns:
    for k, v in candidates_map.items():
        if norm_str(col) == norm_str(k):
            rename_map[col] = v

print("rename_map detectado:", rename_map)
df = df.rename(columns=rename_map)

# 5) ver segurança: garantir colunas renomeadas existem
expected = ["uf", "cod_uf", "cod_mun", "municipio", "populacao_raw"]
miss = [c for c in expected if c not in df.columns]
if miss:
    raise SystemExit(f"Faltam colunas esperadas após renomear: {miss}. Verifique os nomes detectados: {df.columns.tolist()}")

# 6) converter populacao corretamente (sem regex que cria zeros extras)
df["populacao"] = pd.to_numeric(df["populacao_raw"], errors="coerce").round().astype("Int64")

# 7) limpar codigos (remover pontos/virgulas) e formar cod_ibge
df["cod_uf"] = df["cod_uf"].astype(str).str.replace(r"[^0-9]", "", regex=True).str.zfill(2)
df["cod_mun"] = df["cod_mun"].astype(str).str.replace(r"[^0-9]", "", regex=True).str.zfill(5)
df["cod_ibge"] = (df["cod_uf"].fillna("00") + df["cod_mun"].fillna("00000"))
df["cod_ibge"] = pd.to_numeric(df["cod_ibge"], errors="coerce").astype("Int64")

# 8) normalizar nome municipio e uf
df["municipio_clean"] = df["municipio"].astype(str).apply(lambda s: unidecode(s).strip().upper())
df["uf"] = df["uf"].astype(str).str.strip().str.upper()

# 9) remover linhas título/resumo (caso existam)
bad_mask = df["municipio_clean"].isin(["", "UF", "BRASIL", "UNIDADES DA FEDERACAO"])
df_clean = df.loc[~bad_mask].copy()

# 10) checagens rápidas
print("Linhas originais:", len(df), "-> após limpeza:", len(df_clean))
print("Exemplos:")
display(df_clean[["cod_ibge","uf","municipio","municipio_clean","populacao"]].head(10))
print("Registros sem cod_ibge:", df_clean["cod_ibge"].isna().sum())

# 11) salvar CSV limpo
out_dir = Path("data_clean")
out_dir.mkdir(exist_ok=True)
csv_path = out_dir / "estimativa_municipios_2025_clean.csv"
df_clean.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("CSV salvo em:", csv_path)


Colunas originais: ['UF', 'COD. UF', 'COD. MUNIC', 'NOME DO MUNICÍPIO', 'POPULAÇÃO ESTIMADA']


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
0,RO,11.0,15.0,Alta Floresta D'Oeste,22787.0
1,RO,11.0,23.0,Ariquemes,109170.0
2,RO,11.0,31.0,Cabixi,5664.0
3,RO,11.0,49.0,Cacoal,98280.0
4,RO,11.0,56.0,Cerejeiras,16966.0
5,RO,11.0,64.0,Colorado do Oeste,16508.0


rename_map detectado: {'UF': 'uf', 'COD. UF': 'cod_uf', 'COD. MUNIC': 'cod_mun', 'NOME DO MUNICÍPIO': 'municipio', 'POPULAÇÃO ESTIMADA': 'populacao_raw'}
Linhas originais: 5573 -> após limpeza: 5573
Exemplos:


,cod_ibge,uf,municipio,municipio_clean,populacao
0,11000150,RO,Alta Floresta D'Oeste,ALTA FLORESTA D'OESTE,22787
1,11000230,RO,Ariquemes,ARIQUEMES,109170
2,11000310,RO,Cabixi,CABIXI,5664
3,11000490,RO,Cacoal,CACOAL,98280
4,11000560,RO,Cerejeiras,CEREJEIRAS,16966
5,11000640,RO,Colorado do Oeste,COLORADO DO OESTE,16508
6,11000720,RO,Corumbiara,CORUMBIARA,7968
7,11000800,RO,Costa Marques,COSTA MARQUES,13510
8,11000980,RO,Espigão D'Oeste,ESPIGAO D'OESTE,32842
9,11001060,RO,Guajará-Mirim,GUAJARA-MIRIM,43594


Registros sem cod_ibge: 0
CSV salvo em: data_clean\estimativa_municipios_2025_clean.csv


In [47]:
PF_PATH = "REGISTROS_com_categoria_2025.csv"  
SAMPLE_SIZE = 200000  # bytes a ler para detecção

def detect_encoding(path, sample_size=SAMPLE_SIZE):
    try:
        import chardet
        with open(path, "rb") as f:
            raw = f.read(sample_size)
        det = chardet.detect(raw)
        return det.get("encoding"), det
    except Exception:
        try:
            from charset_normalizer import from_bytes
            with open(path, "rb") as f:
                raw = f.read(sample_size)
            res = from_bytes(raw)
            best = res.best()
            return best.encoding, {"confidence": best.chaos}
        except Exception as e:
            print("Nenhuma lib de detecção disponível ou falhou:", e)
            return None, None

def guess_delimiter(sample_text):
    try:
        sniffer = csv.Sniffer()
        dialect = sniffer.sniff(sample_text[:10000])
        return dialect.delimiter
    except Exception:
    
        counts = {d: sample_text.count(d) for d in [",",";","\t","|"]}
        return max(counts, key=counts.get)

def try_read(path):
    
    enc, det = detect_encoding(path)
    print("Detecção (tentativa):", enc, det)
  
    with open(path, "rb") as f:
        raw = f.read(SAMPLE_SIZE)
    try_text = None
    try:
        try_text = raw.decode(enc or "latin-1", errors="replace")
    except Exception:
        try_text = raw.decode("latin-1", errors="replace")
    delim = guess_delimiter(try_text)
    print("Delimitador detectado (estimado):", repr(delim))
   
    enc_try = [enc, "utf-8", "utf-8-sig", "cp1252", "latin-1", "iso-8859-1", "utf-16"]
    tried = []
    for e in [x for x in enc_try if x]:
        for engine in ["c","python"]:
            try:
                print(f"Tentando ler com encoding={e}, engine={engine}, sep={repr(delim)} ...", end=" ")
                df = pd.read_csv(path, encoding=e, sep=delim, engine=engine, low_memory=False)
                print("OK")
                return df, e, delim
            except Exception as ex:
                print("falhou.")
                tried.append((e, engine, str(ex)[:200]))
  
    print("\nTodas tentativas falharam. Tentando leitura bruta para inspeção (sem parse)...")
    text = raw.decode(enc or "latin-1", errors="replace")
    sample_lines = text.splitlines()[:30]
    print("Mostrando 30 primeiras linhas (decodificadas com fallback):\n")
    for i,l in enumerate(sample_lines):
        print(f"{i+1}: {l}")
    raise ValueError(f"Não foi possível ler o CSV automaticamente. Tentativas: {tried[:5]}... (veja as primeiras linhas acima)")


df_pf, used_encoding, used_sep = try_read(PF_PATH)


print("\nLeitura bem-sucedida. shape:", df_pf.shape)
print("Colunas:", df_pf.columns.tolist())
print("\nNulos por coluna (top 20):")
print(df_pf.isnull().sum().sort_values(ascending=False).head(20))

out = Path("REGISTROS_com_categoria_2025_utf8.csv")
df_pf.to_csv(out, index=False, encoding="utf-8-sig")
print("\nArquivo regravado em UTF-8-SIG:", out)
print("Você pode agora usar df_pf no notebook (variável df_pf).")


Detecção (tentativa): cp1252 {'confidence': 0.0}
Delimitador detectado (estimado): ';'
Tentando ler com encoding=cp1252, engine=c, sep=';' ... OK

Leitura bem-sucedida. shape: (19184, 11)
Colunas: ['ANO_EMISSAO', 'MES_MISSAO', 'UF', 'MUNICIPIO', 'STATUS_REGISTRO', 'ESPECIE_ARMA', 'MARCA_ARMA', 'CALIBRE_ARMA', 'SEXO', 'CATEGORIA', 'TOTAL']

Nulos por coluna (top 20):
ANO_EMISSAO        0
MES_MISSAO         0
UF                 0
MUNICIPIO          0
STATUS_REGISTRO    0
ESPECIE_ARMA       0
MARCA_ARMA         0
CALIBRE_ARMA       0
SEXO               0
CATEGORIA          0
TOTAL              0
dtype: int64

Arquivo regravado em UTF-8-SIG: REGISTROS_com_categoria_2025_utf8.csv
Você pode agora usar df_pf no notebook (variável df_pf).


In [49]:
BGE_CLEAN = "data_clean/estimativa_municipios_2025_clean.csv"
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

def normalize_text(s):
    if pd.isna(s): return ""
    return unidecode(str(s)).strip().upper()


print("PF shape:", df_pf.shape)
print("\nNulos por coluna:")
print(df_pf.isnull().sum().sort_values(ascending=False).head(20))
print("\nValores únicos (amostra) - MARCA, ESPECIE, CATEGORIA:")
for c in ["MARCA_ARMA","ESPECIE_ARMA","CATEGORIA"]:
    if c in df_pf.columns:
        print(c, "->", df_pf[c].nunique(), "valores; exemplos:", df_pf[c].unique()[:8])


ibge = pd.read_csv(IBGE_CLEAN, encoding="utf-8-sig", dtype={"cod_ibge":"Int64"})
# garantir coluna normalized
ibge["municipio_clean"] = ibge["municipio_clean"].astype(str).apply(normalize_text)
ibge["uf"] = ibge["uf"].astype(str).str.strip().str.upper()


pf = df_pf.copy()
pf.rename(columns={c:c.strip().upper() for c in pf.columns}, inplace=True)

pf["municipio_clean"] = pf["MUNICIPIO"].astype(str).apply(normalize_text)
pf["uf"] = pf["UF"].astype(str).str.strip().str.upper()
pf["marca_arma_norm"] = pf["MARCA_ARMA"].astype(str).apply(normalize_text) if "MARCA_ARMA" in pf.columns else ""
pf["especie_norm"] = pf["ESPECIE_ARMA"].astype(str).apply(normalize_text) if "ESPECIE_ARMA" in pf.columns else ""
pf["categoria_norm"] = pf["CATEGORIA"].astype(str).apply(normalize_text) if "CATEGORIA" in pf.columns else ""
pf["sexo"] = pf["SEXO"].astype(str).apply(normalize_text) if "SEXO" in pf.columns else ""

# total de armas por registro -> int (coluna "TOTAL")
pf["total_armas"] = pd.to_numeric(
    pf["TOTAL"].astype(str).str.replace(r"[^0-9]", "", regex=True),
    errors="coerce"
).fillna(0).astype(int)



PF shape: (19184, 11)

Nulos por coluna:
ANO_EMISSAO        0
MES_MISSAO         0
UF                 0
MUNICIPIO          0
STATUS_REGISTRO    0
ESPECIE_ARMA       0
MARCA_ARMA         0
CALIBRE_ARMA       0
SEXO               0
CATEGORIA          0
TOTAL              0
dtype: int64

Valores únicos (amostra) - MARCA, ESPECIE, CATEGORIA:
MARCA_ARMA -> 91 valores; exemplos: ['TAURUS ARMAS S.A.                                                     '
 'CBC (COMPANHIA BRASILEIRA DE CARTUCHOS)                               '
 'BOITO (E.R. AMANTINO & CIA)                                           '
 'GLOCK GMBH (ÁUSTRIA)                                                  '
 'SA-KA ARMS                                                            '
 'CZ (CESKOSLOVENSKA ZBROJOVKA)                                         '
 'TOROS ARMS                                                            '
 'HATSAN ARMS                                                           ']
ESPECIE_ARMA -> 10 valores; exe

In [50]:
OUT = Path("output")
OUT.mkdir(exist_ok=True)

# Funções utilitárias
def norm(s):
    if pd.isna(s): return ""
    return unidecode(str(s)).strip().upper()

def normalize_brand(name):
    n = norm(name)
    # detecta TAURUS em vários formatos (TAURUS ARMAS S.A., TAURUS, etc.)
    if "TAURUS" in n:
        return "TAURUS"
    # outras normalizações simples (opcional)
    return n

def normalize_species(s):
    n = norm(s)
    # mapear variantes a PISTOLA / REVOLVER / OUTRAS
    if "PISTOLA" in n or "PISTOL" in n:   # cobre 'Pistola' e variantes
        return "PISTOLA"
    if "REVOLVER" in n or "REVÓLVER" in n:
        return "REVOLVER"
    # espingarda, rifle, carabina -> OUTRAS
    return "OUTRAS"


pf = df_pf.copy()
ibge = pd.read_csv("data_clean/estimativa_municipios_2025_clean.csv", encoding="utf-8-sig", dtype={"cod_ibge":"Int64"})

# limpeza/normalização
pf["municipio_clean"] = pf["MUNICIPIO"].apply(norm)
pf["uf"] = pf["UF"].astype(str).str.strip().str.upper()
pf["marca_norm"] = pf["MARCA_ARMA"].apply(normalize_brand)
pf["especie_norm"] = pf["ESPECIE_ARMA"].apply(normalize_species)
pf["total_armas"] = pd.to_numeric(pf["TOTAL"].astype(str).str.replace(r"[^0-9]","", regex=True), errors="coerce").fillna(0).astype(int)


print("Linhas PF:", len(pf))
print("Marcas únicas (ex.):", list(pd.Series(pf["marca_norm"].unique())[:8]))
print("Espécies mapeadas:", pf["especie_norm"].value_counts().to_dict())
print("Soma total_armas (PF):", pf["total_armas"].sum())


ibge["municipio_clean"] = ibge["municipio_clean"].astype(str).apply(norm)
ibge["uf"] = ibge["uf"].astype(str).str.strip().str.upper()


merged = pf.merge(ibge[["cod_ibge","municipio_clean","populacao","uf"]],
                  left_on=["municipio_clean","uf"], right_on=["municipio_clean","uf"], how="left", suffixes=("","_ibge"))

matched = merged["populacao"].notna().sum()
total = len(merged)
print(f"Match inicial: {matched}/{total} = {matched/total:.1%}")

merged["mun_match_flag"] = merged["populacao"].apply(lambda x: "EXACT" if pd.notna(x) else "NO_MATCH")

agg_total = merged.groupby(["cod_ibge","municipio_clean","uf","populacao"], as_index=False).agg(total_armas=("total_armas","sum"))

espec_pivot = merged.pivot_table(values="total_armas",
                                index=["cod_ibge","municipio_clean","uf","populacao"],
                                columns="especie_norm", aggfunc="sum", fill_value=0).reset_index()

taurus_df = merged[merged["marca_norm"]=="TAURUS"].groupby(["cod_ibge","municipio_clean","uf","populacao"], as_index=False).agg(taurus_total=("total_armas","sum"))

df_mun = agg_total.merge(espec_pivot, on=["cod_ibge","municipio_clean","uf","populacao"], how="left").merge(taurus_df, on=["cod_ibge","municipio_clean","uf","populacao"], how="left").fillna(0)

df_mun["armas_por_100k"] = df_mun.apply(lambda r: (r["total_armas"]/r["populacao"]*100000) if r["populacao"] and r["populacao"]>0 else None, axis=1)
if "PISTOLA" in df_mun.columns:
    df_mun["pct_pistola"] = df_mun["PISTOLA"] / df_mun["total_armas"].replace({0:pd.NA})
else:
    df_mun["pct_pistola"] = pd.NA

df_mun["taurus_total"] = df_mun["taurus_total"].fillna(0).astype(int)
df_mun["pct_taurus"] = df_mun.apply(lambda r: (r["taurus_total"]/r["total_armas"]) if r["total_armas"]>0 else None, axis=1)

top3 = df_mun.nlargest(3, "total_armas")["cod_ibge"].tolist()
df_mun["is_top3_pais"] = df_mun["cod_ibge"].isin(top3)

capitals_map = {
 "AC":"RIO BRANCO","AL":"MACEIO","AP":"MACAPA","AM":"MANAUS","BA":"SALVADOR","CE":"FORTALEZA","DF":"BRASILIA",
 "ES":"VITORIA","GO":"GOIANIA","MA":"SAO LUIS","MT":"CUIABA","MS":"CAMPO GRANDE","MG":"BELO HORIZONTE",
 "PA":"BELEM","PB":"JOAO PESSOA","PR":"CURITIBA","PE":"RECIFE","PI":"TERESINA","RJ":"RIO DE JANEIRO",
 "RN":"NATAL","RS":"PORTO ALEGRE","RO":"PORTO VELHO","RR":"BOA VISTA","SC":"FLORIANOPOLIS","SP":"SAO PAULO",
 "SE":"ARACAJU","TO":"PALMAS"
}
capitals_map = {k: norm(v) for k,v in capitals_map.items()}
df_mun["is_capital"] = df_mun.apply(lambda r: True if r["uf"] in capitals_map and r["municipio_clean"]==capitals_map[r["uf"]] else False, axis=1)

agg_uf = df_mun.groupby("uf", as_index=False).agg(total_armas_uf=("total_armas","sum"), pop_uf=("populacao","sum"))
agg_uf["armas_por_100k_uf"] = agg_uf.apply(lambda r: (r["total_armas_uf"]/r["pop_uf"]*100000) if r["pop_uf"]>0 else None, axis=1)
agg_uf = agg_uf.sort_values("total_armas_uf", ascending=False).reset_index(drop=True)
agg_uf["rank_uf_total"] = agg_uf["total_armas_uf"].rank(method="dense", ascending=False).astype(int)

orig_sum = pf["total_armas"].sum()
mun_sum = df_mun["total_armas"].sum()
uf_sum = agg_uf["total_armas_uf"].sum()

print("Soma original PF:", orig_sum)
print("Soma agregado município:", mun_sum)
print("Soma agregado UF:", uf_sum)
print("Perda absoluta (orig - muni):", orig_sum - mun_sum)

df_mun.to_csv(OUT/"armas_por_municipio.csv", index=False, encoding="utf-8-sig")
agg_uf.to_csv(OUT/"armas_por_uf.csv", index=False, encoding="utf-8-sig")
merged.to_csv(OUT/"pf_merged_with_ibge_raw.csv", index=False, encoding="utf-8-sig")

print("Arquivos salvos em:", OUT)


Linhas PF: 19184
Marcas únicas (ex.): ['TAURUS', 'CBC (COMPANHIA BRASILEIRA DE CARTUCHOS)', 'BOITO (E.R. AMANTINO & CIA)', 'GLOCK GMBH (AUSTRIA)', 'SA-KA ARMS', 'CZ (CESKOSLOVENSKA ZBROJOVKA)', 'TOROS ARMS', 'HATSAN ARMS']
Espécies mapeadas: {'PISTOLA': 12576, 'OUTRAS': 4875, 'REVOLVER': 1733}
Soma total_armas (PF): 25438
Match inicial: 19153/19184 = 99.8%
Soma original PF: 25438
Soma agregado município: 25405
Soma agregado UF: 25405
Perda absoluta (orig - muni): 33
Arquivos salvos em: output


In [52]:
no_match = merged[merged["mun_match_flag"]=="NO_MATCH"]
len(no_match), no_match.head(10)

no_match.to_csv("output/pf_no_match_sample.csv", index=False, encoding="utf-8-sig")


In [53]:
# top 10 por total_armas (municipal agregado)
top10 = df_mun.sort_values("total_armas", ascending=False).head(10)
top10[["cod_ibge","municipio_clean","uf","total_armas","armas_por_100k","taurus_total"]]


,cod_ibge,municipio_clean,uf,total_armas,armas_por_100k,taurus_total
1495,53001080,BRASILIA,DF,604,20.154166,366
675,31062000,BELO HORIZONTE,MG,490,20.282532,332
2283,350503080,SAO PAULO,SP,429,3.603540,259
828,33045570,RIO DE JANEIRO,RJ,406,6.032036,155
1484,52087070,GOIANIA,GO,386,25.677596,252
1279,50027040,CAMPO GRANDE,MS,379,39.360961,297
902,35095020,CAMPINAS,SP,328,27.610032,239
91,13026030,MANAUS,AM,310,13.456426,241
1697,290274080,SALVADOR,BA,270,10.529583,157
109,14001000,BOA VISTA,RR,263,54.173524,179


In [54]:
# capitais marcadas
df_mun[df_mun["is_capital"]].sort_values("uf")[["uf","municipio_clean","total_armas","armas_por_100k"]].head(30)


,uf,municipio_clean,total_armas,armas_por_100k
64,AC,RIO BRANCO,128,32.904800
542,AL,MACEIO,148,14.875089
91,AM,MANAUS,310,13.456426
188,AP,MACAPA,143,29.202983
1697,BA,SALVADOR,270,10.529583
365,CE,FORTALEZA,234,9.075103
1495,DF,BRASILIA,604,20.154166
770,ES,VITORIA,44,12.813867
1484,GO,GOIANIA,386,25.677596
1521,MA,SAO LUIS,129,11.843392


In [55]:
[s for s in df_mun.columns if s not in ["cod_ibge","municipio_clean","uf","populacao","total_armas","armas_por_100k","taurus_total","pct_taurus","pct_pistola","is_top3_pais","is_capital"]][:30]

taurus_by_uf = merged[merged["marca_norm"]=="TAURUS"].groupby("uf", as_index=False).agg(taurus_total=("total_armas","sum"))
taurus_by_uf = taurus_by_uf.merge(agg_uf, on="uf", how="right").fillna(0)
taurus_by_uf["pct_taurus_uf"] = taurus_by_uf["taurus_total"] / taurus_by_uf["total_armas_uf"]
taurus_by_uf.sort_values("taurus_total", ascending=False).head(10)


,uf,taurus_total,total_armas_uf,pop_uf,armas_por_100k_uf,rank_uf_total,pct_taurus_uf
0,RS,2112,3956,10789388.0,36.665657,1,0.533873
1,SP,1883,2922,42922595.0,6.807603,2,0.644422
2,MG,1585,2611,17665416.0,14.780292,3,0.607047
3,SC,1189,2111,7789373.0,27.101026,4,0.563240
4,GO,1075,1665,7137782.0,23.326574,5,0.645646
5,ES,794,1246,4112559.0,30.297438,6,0.637239
7,RJ,611,1046,16989534.0,6.156732,8,0.584130
8,BA,609,925,10320320.0,8.962900,9,0.658378
6,PR,596,1171,9885672.0,11.845426,7,0.508967
10,MS,465,694,2716430.0,25.548238,11,0.670029


In [56]:
(df_mun[df_mun["populacao"]==0]).shape

df_mun.sort_values("armas_por_100k", ascending=False).head(20)[["cod_ibge","municipio_clean","uf","total_armas","populacao","armas_por_100k"]]


,cod_ibge,municipio_clean,uf,total_armas,populacao,armas_por_100k
2545,420191500,VARGEM,SC,12,2613.0,459.242250
1253,43091590,GRAMADO XAVIER,RS,14,3350.0,417.910448
1157,43041010,CAMPOS BORGES,RS,15,3695.0,405.953992
2434,420100500,MACIEIRA,SC,7,1796.0,389.755011
1102,43005050,ALPESTRE,RS,26,7235.0,359.364202
6,11000720,CORUMBIARA,RO,27,7968.0,338.855422
2751,430204530,SERIO,RS,7,2089.0,335.088559
1222,43070540,ERNESTINA,RS,10,3097.0,322.893122
2495,420150750,RIQUEZA,SC,15,4828.0,310.687655
975,42004080,AGUA DOCE,SC,20,6587.0,303.628359


In [57]:
df_mun.to_csv("output/municipios_armas_ibge_2025.csv", index=False, encoding="utf-8-sig")

In [61]:
df_mun["localizacao_completa"] = df_mun["municipio_clean"] + ", " + df_mun["uf"] + ", Brasil"


In [62]:
df_mun[["municipio_clean", "uf", "localizacao_completa"]].head(10)

,municipio_clean,uf,localizacao_completa
0,ALTA FLORESTA D'OESTE,RO,"ALTA FLORESTA D'OESTE, RO, Brasil"
1,ARIQUEMES,RO,"ARIQUEMES, RO, Brasil"
2,CABIXI,RO,"CABIXI, RO, Brasil"
3,CACOAL,RO,"CACOAL, RO, Brasil"
4,CEREJEIRAS,RO,"CEREJEIRAS, RO, Brasil"
5,COLORADO DO OESTE,RO,"COLORADO DO OESTE, RO, Brasil"
6,CORUMBIARA,RO,"CORUMBIARA, RO, Brasil"
7,COSTA MARQUES,RO,"COSTA MARQUES, RO, Brasil"
8,ESPIGAO D'OESTE,RO,"ESPIGAO D'OESTE, RO, Brasil"
9,GUAJARA-MIRIM,RO,"GUAJARA-MIRIM, RO, Brasil"


In [66]:
df_mun.to_csv("municipios_armas_ibge_2025.csv", index=False, encoding="utf-8-sig")

In [67]:
pd.read_csv("municipios_armas_ibge_2025.csv").head()

,cod_ibge,municipio_clean,uf,populacao,total_armas,OUTRAS,PISTOLA,REVOLVER,taurus_total,armas_por_100k,pct_pistola,pct_taurus,is_top3_pais,is_capital,localizacao_completa
0,11000150,ALTA FLORESTA D'OESTE,RO,22787.0,55,30,20,5,22,241.365691,0.363636,0.400000,False,False,"ALTA FLORESTA D'OESTE, RO, Brasil"
1,11000230,ARIQUEMES,RO,109170.0,55,21,31,3,30,50.380141,0.563636,0.545455,False,False,"ARIQUEMES, RO, Brasil"
2,11000310,CABIXI,RO,5664.0,17,14,3,0,3,300.141243,0.176471,0.176471,False,False,"CABIXI, RO, Brasil"
3,11000490,CACOAL,RO,98280.0,22,10,11,1,8,22.385022,0.500000,0.363636,False,False,"CACOAL, RO, Brasil"
4,11000560,CEREJEIRAS,RO,16966.0,38,17,19,2,19,223.977366,0.500000,0.500000,False,False,"CEREJEIRAS, RO, Brasil"


In [68]:
df_mun.columns


Index(['cod_ibge', 'municipio_clean', 'uf', 'populacao', 'total_armas',
       'OUTRAS', 'PISTOLA', 'REVOLVER', 'taurus_total', 'armas_por_100k',
       'pct_pistola', 'pct_taurus', 'is_top3_pais', 'is_capital',
       'localizacao_completa'],
      dtype='object')

In [70]:
df_mun.to_csv("municipios_armas_ibge_2025.csv", index=False, encoding="utf-8-sig")

In [74]:
import os
print(os.path.abspath("municipios_armas_ibge_2025.csv"))


C:\projetos_faculdade\dados_pi_taurus\municipios_armas_ibge_2025.csv


In [92]:
# Top-3 municípios por estado
df_mun = df_mun.copy() 
df_mun["rank_uf"] = df_mun.groupby("uf")["total_armas"].rank(method="first", ascending=False)
df_mun["is_top3_uf"] = df_mun["rank_uf"] <= 3
print(df_mun[["uf","municipio_clean","total_armas","rank_uf","is_top3_uf"]].head(10))

   uf        municipio_clean  total_armas  rank_uf  is_top3_uf
0  RO  ALTA FLORESTA D'OESTE           55      3.0        True
1  RO              ARIQUEMES           55      4.0       False
2  RO                 CABIXI           17     13.0       False
3  RO                 CACOAL           22     11.0       False
4  RO             CEREJEIRAS           38      5.0       False
5  RO      COLORADO DO OESTE           36      6.0       False
6  RO             CORUMBIARA           27      9.0       False
7  RO          COSTA MARQUES            2     46.0       False
8  RO        ESPIGAO D'OESTE            2     47.0       False
9  RO          GUAJARA-MIRIM            9     28.0       False


In [89]:
# Salvar CSV atualizado
df_mun.to_csv(OUT/"armas_por_municipio_top3_uf.csv", index=False, encoding="utf-8-sig")


In [90]:
test = pd.read_csv("municipios_armas_ibge_2025.csv")


In [91]:
test.columns

Index(['cod_ibge', 'municipio_clean', 'uf', 'populacao', 'total_armas',
       'OUTRAS', 'PISTOLA', 'REVOLVER', 'taurus_total', 'armas_por_100k',
       'pct_pistola', 'pct_taurus', 'is_top3_pais', 'is_capital',
       'localizacao_completa'],
      dtype='object')